In [1]:
import pandas as pd 
import numpy as np 
from pathlib import Path
from src.data.functions import json_to_dict

In [12]:
top_dir = Path().absolute()
PARENT_FILE_PATH = top_dir / "models" / "2025-10-06 TSLP Platform Fit Model"
model_config = json_to_dict(Path(PARENT_FILE_PATH, "TSLP_PF_model_parameters.json"))

a_matrix = np.array(model_config["a_matrix"])[0]
b_matrix = np.array(model_config["b_matrix"])[0]

In [13]:
import numpy as np
from scipy.linalg import solve_lyapunov, svdvals
from scipy.signal import StateSpace, lsim

class InputSensitivityAnalyzer:
    def __init__(self, A, B, C, D, input_names=None, output_names=None):
        self.A, self.B, self.C, self.D = A, B, C, D
        self.n_states = A.shape[0]
        self.n_inputs = B.shape[1]
        self.n_outputs = C.shape[0]
        self.input_names = input_names or [f"u{i}" for i in range(self.n_inputs)]
        self.output_names = output_names or [f"y{i}" for i in range(self.n_outputs)]

    def dc_gain(self):
        """Steady-state gain matrix K: output change per unit input change."""
        return -self.C @ np.linalg.solve(self.A, self.B) + self.D

    def controllability_gramians_per_input(self):
        """Solve Lyapunov equation per input channel."""
        gramians = []
        for j in range(self.n_inputs):
            bj = self.B[:, j:j+1]
            # AW + WA^T + bb^T = 0  =>  W = solve_lyapunov(A, -bb^T)
            Wj = solve_lyapunov(self.A, -bj @ bj.T)
            gramians.append(Wj)
        return gramians

    def input_importance(self):
        """Rank inputs by multiple criteria."""
        K = self.dc_gain()
        gramians = self.controllability_gramians_per_input()

        results = []
        for j in range(self.n_inputs):
            dc_norm = np.linalg.norm(K[:, j])
            gram_trace = np.trace(gramians[j])
            gram_max_eig = np.max(np.real(np.linalg.eigvals(gramians[j])))
            results.append({
                'input': self.input_names[j],
                'dc_gain_norm': dc_norm,
                'gramian_trace': gram_trace,
                'gramian_max_eigenvalue': gram_max_eig,
            })

        # Sort by gramian trace (total controllability energy)
        return sorted(results, key=lambda x: x['gramian_trace'], reverse=True)

    def frequency_response_per_input(self, freqs):
        """||G_j(jw)|| across frequency range for each input."""
        I = np.eye(self.n_states)
        results = {name: [] for name in self.input_names}

        for omega in freqs:
            resolvent = np.linalg.solve(1j * omega * I - self.A, self.B)
            G_jw = self.C @ resolvent + self.D
            for j in range(self.n_inputs):
                results[self.input_names[j]].append(np.linalg.norm(G_jw[:, j]))

        return freqs, results

In [14]:
c_matrix = np.identity(a_matrix.shape[0])
d_matrix = np.zeros([a_matrix.shape[0], b_matrix.shape[1]])

sensitivity = InputSensitivityAnalyzer(a_matrix, b_matrix, c_matrix, d_matrix)
importance = sensitivity.input_importance()

importance

[{'input': 'u1',
  'dc_gain_norm': np.float64(2.4838018403752784),
  'gramian_trace': np.float64(0.3814368643980188),
  'gramian_max_eigenvalue': np.float64(0.18270911167227433)},
 {'input': 'u0',
  'dc_gain_norm': np.float64(0.8833419874433224),
  'gramian_trace': np.float64(0.10259027228622573),
  'gramian_max_eigenvalue': np.float64(0.08129451079568412)},
 {'input': 'u2',
  'dc_gain_norm': np.float64(0.8040969709251181),
  'gramian_trace': np.float64(0.04402197106551089),
  'gramian_max_eigenvalue': np.float64(0.025305279556097147)},
 {'input': 'u3',
  'dc_gain_norm': np.float64(0.4766793102522386),
  'gramian_trace': np.float64(0.03412202749308124),
  'gramian_max_eigenvalue': np.float64(0.027902144296634148)}]